<a href="https://colab.research.google.com/github/elkins/synth-pdb/blob/main/examples/interactive_tutorials/saxs_shape_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Shape Decoder: What X-Rays Reveal About Protein Architecture

**Small-Angle X-ray Scattering (SAXS)** measures the overall shape of a protein in solution —
no crystals required, no freezing, just the native hydrated protein in a capillary tube.

The scattered X-ray intensity profile I(q) encodes the protein's shape in a compact 1D signal.
Three tools decode it:

| Tool | What it reveals |
|---|---|
| **Guinier plot** | Size (radius of gyration R_g) |
| **Kratky plot** | Compactness vs. disorder |
| **P(r) distribution** | Overall shape fingerprint |

> **Key insight**: If you can't crystallize your protein — and most IDPs never crystallize —
> SAXS is often the only technique that gives you its shape in solution.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q synth-pdb synth-saxs biotite matplotlib numpy
else:
    sys.path.insert(0, os.path.abspath("../../"))

import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import biotite.structure.io.pdb as bpdb
from synth_pdb.generator import generate_pdb_content
from synth_saxs import calculate_saxs_profile, calculate_radius_of_gyration, calculate_p_dist

print("✅ Environment ready")

## Three Architectures, Three Signatures

We'll generate three distinct protein architectures and compare their SAXS signatures:

| Architecture | Sequence | Expected shape |
|---|---|---|
| **Compact α-helix bundle** | AKA-repeat (30 aa) | Small R_g, bell-shaped Kratky |
| **Extended β-sheet** | Poly-Val (20 aa) | Larger R_g, broader P(r) |
| **Disordered IDP** | GS-repeat (30 aa) | Largest R_g, rising Kratky |

The GS-repeat (glycine-serine) is a **canonical disordered linker** — no hydrophobic core,
no secondary structure, maximum conformational freedom.

In [ ]:
def parse_pdb(pdb_str):
    return bpdb.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)


print("Generating protein structures...")
structs = {}
pdbs = {}

configs = [
    ("Compact Helix", "AAAAAKKKKAAAAAKKKKAAAAAKKKKAAAA", "alpha", "royalblue"),
    ("Extended Sheet", "VVVVVVVVVVVVVVVVVVVV", "beta", "tomato"),
    ("Disordered IDP", "GSGSGSGSGSGSGSGSGSGSGSGSGSGSGS", "random", "mediumseagreen"),
]

for name, seq, conf, _ in configs:
    pdb_str = generate_pdb_content(sequence_str=seq, conformation=conf, seed=42)
    pdbs[name] = pdb_str
    structs[name] = parse_pdb(pdb_str)
    print(f"  {name}: {len(seq)} residues, {structs[name].array_length()} atoms")

print("\nCalculating SAXS profiles and shape metrics...")
Q_MIN, Q_MAX, N_PTS = 0.01, 0.5, 60

saxs = {}
rg_map = {}
pr_map = {}

for name, _, _, _ in configs:
    s = structs[name]
    q, I = calculate_saxs_profile(s, q_min=Q_MIN, q_max=Q_MAX, n_points=N_PTS)
    saxs[name] = (q, I)
    rg_map[name] = calculate_radius_of_gyration(s)
    r, p = calculate_p_dist(s, bins=50)
    pr_map[name] = (r, p)

print("\n📐 Radius of Gyration comparison:")
for name, _, _, _ in configs:
    print(f"   {name:20s}: Rg = {rg_map[name]:.2f} Å")

## The Four-Panel SAXS Dashboard

**Guinier analysis**: In the low-q region where q·R_g < 1.3, the scattering follows:

$$\ln I(q) \approx \ln I_0 - \frac{R_g^2}{3} q^2$$

A straight line on a ln(I) vs q² plot confirms the sample is monodisperse and gives R_g
from the slope. A downward curve means aggregation; an upward curve means repulsion.

**Kratky plot** (q²I vs q): The shape of this curve is the folding sensor:
- Compact globular protein → **bell-shaped peak** (rises then falls)
- Intrinsically disordered → **monotonically rising** (Gaussian chain behaviour)

In [ ]:
COLOR_MAP = {
    "Compact Helix": "royalblue",
    "Extended Sheet": "tomato",
    "Disordered IDP": "mediumseagreen",
}

plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(15, 11))
gs = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)
ax1 = fig.add_subplot(gs[0, 0])  # SAXS profiles
ax2 = fig.add_subplot(gs[0, 1])  # Guinier
ax3 = fig.add_subplot(gs[1, 0])  # Kratky
ax4 = fig.add_subplot(gs[1, 1])  # P(r)

for name, _, _, _ in configs:
    q, I = saxs[name]
    r, p = pr_map[name]
    Rg = rg_map[name]
    col = COLOR_MAP[name]

    # ── Panel 1: log-scale SAXS profile ──────────────────────────────────
    ax1.semilogy(q, I, color=col, lw=2.2, label=f"{name} (Rg={Rg:.1f}Å)")

    # ── Panel 2: Guinier plot ─────────────────────────────────────────────
    mask = (q * Rg < 1.3) & (I > 0)
    q2g = q[mask] ** 2
    lnIg = np.log(I[mask])
    ax2.scatter(q2g, lnIg, color=col, s=22, alpha=0.8, zorder=3)
    if mask.sum() >= 2:
        c = np.polyfit(q2g, lnIg, 1)
        x_fit = np.linspace(q2g.min(), q2g.max(), 50)
        Rg_fit = np.sqrt(-c[0] * 3) if -c[0] > 0 else Rg
        ax2.plot(
            x_fit, np.polyval(c, x_fit), "--", color=col, lw=1.5, label=f"{name} Rg={Rg_fit:.1f}Å"
        )

    # ── Panel 3: Kratky plot ──────────────────────────────────────────────
    ax3.plot(q, q**2 * I, color=col, lw=2.2, label=name)
    ax3.fill_between(q, q**2 * I, alpha=0.10, color=col)

    # ── Panel 4: P(r) ─────────────────────────────────────────────────────
    p_norm = p / (p.max() if p.max() > 0 else 1.0)
    ax4.plot(r, p_norm, color=col, lw=2.2, label=name)
    ax4.fill_between(r, p_norm, alpha=0.12, color=col)

# ── Formatting ────────────────────────────────────────────────────────────
ax1.set_xlabel("q (Å⁻¹)", fontsize=11)
ax1.set_ylabel("I(q) (log scale)", fontsize=11)
ax1.set_title("SAXS Profiles", fontsize=13, fontweight="bold")
ax1.legend(fontsize=9.5)
ax1.set_xlim(Q_MIN, Q_MAX)

ax2.set_xlabel("q² (Å⁻²)", fontsize=11)
ax2.set_ylabel("ln I(q)", fontsize=11)
ax2.set_title("Guinier Analysis\n(Guinier region: q·Rg < 1.3)", fontsize=13, fontweight="bold")
ax2.legend(fontsize=9.5)

ax3.set_xlabel("q (Å⁻¹)", fontsize=11)
ax3.set_ylabel("q²·I(q)", fontsize=11)
ax3.set_title(
    "Kratky Plot — The Folding Sensor\nBell = compact | Rising = disordered",
    fontsize=13,
    fontweight="bold",
)
ax3.legend(fontsize=9.5)
ax3.set_xlim(Q_MIN, Q_MAX)

ax4.set_xlabel("r (Å)", fontsize=11)
ax4.set_ylabel("P(r) (normalised)", fontsize=11)
ax4.set_title(
    "P(r) Pair-Distance Distribution\nShape fingerprint in real space",
    fontsize=13,
    fontweight="bold",
)
ax4.legend(fontsize=9.5)

fig.suptitle(
    "SAXS Shape Fingerprints: Helix · Sheet · Disordered", fontsize=16, fontweight="bold", y=1.01
)
plt.savefig("saxs_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## The Kratky Plot: A Folding Sensor

The physics behind the Kratky plot:

- For a **compact globular protein** (scattering from a sphere), I(q) ∝ q⁻⁴ at large q,
  so q²·I(q) → 0. This gives the characteristic **bell shape**.

- For an **ideal Gaussian chain** (fully disordered IDP), I(q) ∝ q⁻², so q²·I(q) = constant.
  This gives the **monotonically rising plateau**.

- Real IDPs fall between these extremes — a gently rising curve signals partial disorder.

**This single plot distinguishes folded from disordered proteins without any atomic model** —
a remarkable achievement from just X-ray scattering data.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

names = [n for n, *_ in configs]
rg_vals = [rg_map[n] for n in names]
colors = [COLOR_MAP[n] for n in names]
err = [rg * 0.05 for rg in rg_vals]  # ±5% typical experimental uncertainty

bars = ax.bar(names, rg_vals, color=colors, edgecolor="white", linewidth=1.5, width=0.55, zorder=3)
ax.errorbar(
    names,
    rg_vals,
    yerr=err,
    fmt="none",
    color="#333",
    capsize=6,
    capthick=2,
    elinewidth=2,
    zorder=4,
)

for bar, val in zip(bars, rg_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.5,
        f"{val:.1f} Å",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=12,
    )

ax.set_ylabel("Radius of Gyration R_g (Å)", fontsize=12)
ax.set_title(
    "Architecture Determines Size\nRadius of Gyration from SAXS", fontsize=14, fontweight="bold"
)
ax.set_ylim(0, max(rg_vals) * 1.25)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("saxs_rg_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Real-World Impact: SAXS in Drug Discovery

SAXS has revealed that many disease-linked proteins are **intrinsically disordered**:

| Protein | Disease | SAXS finding |
|---|---|---|
| **α-Synuclein** | Parkinson's | IDP with rising Kratky — no stable fold |
| **Tau** | Alzheimer's | Extended IDP, Rg >> folded equivalent |
| **p53 TAD** | Cancer | Disordered transactivation domain |
| **FUS/TDP-43** | ALS | Disordered low-complexity regions |

These proteins are "undruggable" by classical small-molecule targeting because they have no
stable binding pockets — the Kratky plot told us this long before high-resolution structures
were available.

**Modern approaches** (PROTACs, molecular glues, RNA aptamers) specifically exploit the
disordered nature that the rising Kratky curve reveals.